# Beto

Implementación de BETO (BERT en español) para el análisis semántico de reseñas turísticas de Costa Rica. En este notebook se implementan embeddings contextuales, análisis de polisemia, búsqueda semántica y Masked Language Model utilizando el modelo oficial BETO de Hugging Face.

In [ ]:
!pip install transformers torch spacy

1. Importación de librerías

In [13]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    pipeline
)

from sklearn.metrics.pairwise import cosine_similarity

2. Cargar el corpus

In [14]:
df = pd.read_csv("../data/processed/reseñas_pos_tagged_P2.csv")

df.head()

,reseña,calificación,fuente,nombre,fecha,tokens_spacy,tokens_nltk
0,excelente lugar camas aire acondicinado todo n...,5,google_places_api,sky hostel fortuna,2026-03-30,"[('excelente', 'ADJ'), ('lugar', 'NOUN'), ('ca...","[('excelente', 'aq0cs0'), ('lugar', 'ncms000')..."
1,la cabina muy bien equipada y limpia pero la s...,1,google_places_api,cabinas guayabón,2026-07-02,"[('la', 'DET'), ('cabina', 'NOUN'), ('muy', 'A...","[('la', 'da0fs0'), ('cabina', None), ('muy', '..."
2,definitivamente un lugar muy tranquilo para de...,4,google_places_api,cabinas guayabón,2025-10-21,"[('definitivamente', 'ADV'), ('un', 'DET'), ('...","[('definitivamente', 'rg'), ('un', 'di0ms0'), ..."
3,una experiencia agradable con ubicación muy bu...,4,google_places_api,cabinas guayabón,2026-01-18,"[('una', 'DET'), ('experiencia', 'NOUN'), ('ag...","[('una', 'di0fs0'), ('experiencia', 'ncfs000')..."
4,sobre las aguas termales recomendado están sep...,4,google_places_api,cabinas las termalitas,2026-03-17,"[('sobre', 'ADP'), ('las', 'DET'), ('aguas', '...","[('sobre', 'sps00'), ('las', 'da0fp0'), ('agua..."


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1968 entries, 0 to 1967
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   reseña        1968 non-null   str  
 1   calificación  1968 non-null   int64
 2   fuente        1968 non-null   str  
 3   nombre        1968 non-null   str  
 4   fecha         1968 non-null   str  
 5   tokens_spacy  1968 non-null   str  
 6   tokens_nltk   1968 non-null   str  
dtypes: int64(1), str(6)
memory usage: 5.1 MB


3. Cargar BETO desde Hugging Face

In [16]:
modelo = "dccuchile/bert-base-spanish-wwm-cased"

tokenizer = AutoTokenizer.from_pretrained(modelo)

model = AutoModel.from_pretrained(modelo)

print("BETO cargado correctamente.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BETO cargado correctamente.


4. Entendiendo cómo funciona BETO

In [17]:
texto = "Me encantó visitar el Volcán Arenal."

tokens = tokenizer(texto)

tokens

{'input_ids': [4, 1369, 25921, 9742, 1040, 5288, 10387, 27191, 30938, 1009, 5], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [18]:
tokenizer.convert_ids_to_tokens(tokens["input_ids"])

['[CLS]',
 'Me',
 'encantó',
 'visitar',
 'el',
 'Vol',
 '##cán',
 'Arena',
 '##l',
 '.',
 '[SEP]']

5. Generar embeddings para todas las reseñas

In [19]:
# Función para obtener el embedding

def obtener_embedding(texto):

    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Promedio de todos los tokens
    embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding.numpy()[0]

In [20]:
print("Generando embeddings...")

df["embedding"] = df["reseña"].apply(obtener_embedding)

print("Embeddings generados.")

Generando embeddings...
Embeddings generados.


In [21]:
print(df["embedding"].iloc[0].shape)

(768,)


6. Polisemia

In [23]:
#7.1 Buscar las palabras más frecuentes del corpus
from collections import Counter
import re
import spacy

# Cargar modelo de spaCy
nlp = spacy.load("es_core_news_md")

texto = " ".join(df["reseña"].astype(str))

# Tokenizar
doc = nlp(texto)

# Eliminar stopwords, signos y palabras muy cortas
palabras = [
    token.lemma_.lower()
    for token in doc
    if not token.is_stop
    and not token.is_punct
    and not token.like_num
    and len(token.text) > 2
]

conteo = Counter(palabras)

frecuentes = pd.DataFrame(
    conteo.most_common(50),
    columns=["palabra", "frecuencia"]
)

frecuentes

,palabra,frecuencia
0,lugar,1131
1,comida,721
2,excelente,627
3,habitación,501
4,servicio,415
5,increíble,366
6,amable,358
7,personal,352
8,precio,336
9,limpio,292


In [24]:
#7.2 Elegir una palabra
palabra = "servicio"

#Más palabras
#aire = aire acondicionado, aire libre
#café = bebida, cafetería
#cocina = cocina (habitación), cocina (acción de cocinar)
#lugar =lugar físico, lugar turístico

In [25]:
#7.3 Buscar reseñas donde aparece esa palabra
def buscar_contextos(palabra, cantidad=5):

    resultados = df[
        df["reseña"].str.contains(
            rf"\b{palabra}\b",
            case=False,
            regex=True,
            na=False
        )
    ]

    return resultados.head(cantidad)

In [26]:
#7.4 Mostrar los ejemplos
ejemplos = buscar_contextos(palabra)

for i, fila in ejemplos.iterrows():

    print("="*80)
    print(f"Lugar: {fila['nombre']}")
    print()
    print(fila["reseña"])
    print()

Lugar: cabinas guayabón

la cabina muy bien equipada y limpia pero la señora que atiende le recomiendo hacer un cursito de servicio al cliente

Lugar: hospedaje vacacional cr

un muy buen lugar equipado con todo lo necesario para tu estadía en manuel antonio los chicos anfitriones son muy serviciales recomendado total hay servicio de wifi y la tranquilidad de la zona es particularmente especial para un buen descanso cerca a las playas y a los supermercado estacionamiento amplio y vigilado

Lugar: hospedaje vacacional cr

si hacen reserva por booking es falsa ya que ellos no tienen acceso a la plataforma baños se taquean constantemente y hay fugas de agua en las paredes el servicio es bueno la atención es buena solamente que si buscan comodidad tal vez no sea el lugar por esas dos cosas que tiene mal

Lugar: hotel vista real manuel antonio

a mi me llamó mucho la atención desde el momento en que hice consultas y la muchacha me atendió con mucha amabilidad cuando llegamos al lugar la muc

In [27]:
#7.5 Escoger dos reseñas
texto1 = ejemplos.iloc[0]["reseña"]
texto2 = ejemplos.iloc[1]["reseña"]

In [28]:
#7.6 Comparar los embeddings
def comparar_contextos(texto1, texto2):

    emb1 = obtener_embedding(texto1)
    emb2 = obtener_embedding(texto2)

    similitud = cosine_similarity(
        emb1.reshape(1,-1),
        emb2.reshape(1,-1)
    )[0][0]

    print("="*70)

    print("RESEÑA 1\n")
    print(texto1)

    print("\n")

    print("RESEÑA 2\n")
    print(texto2)

    print("\n")

    print("Similitud coseno:", round(similitud,4))

    if similitud > 0.75:
        print("\nBETO considera ambos contextos muy similares.")

    elif similitud > 0.55:
        print("\nBETO detecta diferencias moderadas entre los contextos.")

    else:
        print("\nBETO considera que los contextos son bastante distintos.")

In [29]:
comparar_contextos(texto1, texto2)

RESEÑA 1

la cabina muy bien equipada y limpia pero la señora que atiende le recomiendo hacer un cursito de servicio al cliente


RESEÑA 2

un muy buen lugar equipado con todo lo necesario para tu estadía en manuel antonio los chicos anfitriones son muy serviciales recomendado total hay servicio de wifi y la tranquilidad de la zona es particularmente especial para un buen descanso cerca a las playas y a los supermercado estacionamiento amplio y vigilado


Similitud coseno: 0.9361

BETO considera ambos contextos muy similares.


La palabra servicio fue evaluada en dos contextos diferentes: uno relacionado con la atención al cliente y otro con el servicio de wifi ofrecido por el hospedaje. BETO obtuvo una similitud coseno de 0.9361, indicando que considera ambos usos muy cercanos semánticamente. Esto puede deberse a que ambos aparecen en el mismo dominio turístico y representan servicios ofrecidos por un establecimiento.

In [56]:
#7.2 Elegir una palabra
palabra = "atención"

In [57]:
#7.3 Buscar reseñas donde aparece esa palabra
def buscar_contextos(palabra, cantidad=10):

    resultados = df[
        df["reseña"].str.contains(
            rf"\b{palabra}\b",
            case=False,
            regex=True,
            na=False
        )
    ]

    return resultados.head(cantidad)

In [58]:
#7.4 Mostrar los ejemplos
ejemplos = buscar_contextos(palabra)

for i, fila in ejemplos.iterrows():

    print("="*80)
    print(f"Lugar: {fila['nombre']}")
    print()
    print(fila["reseña"])
    print()

Lugar: hospedaje vacacional cr

si hacen reserva por booking es falsa ya que ellos no tienen acceso a la plataforma baños se taquean constantemente y hay fugas de agua en las paredes el servicio es bueno la atención es buena solamente que si buscan comodidad tal vez no sea el lugar por esas dos cosas que tiene mal

Lugar: hospedaje vacacional cr

me encantó el lugar enoc se portó genial me gustó mucho la atención las instalaciones todo muy lindo

Lugar: hotel vista real manuel antonio

a mi me llamó mucho la atención desde el momento en que hice consultas y la muchacha me atendió con mucha amabilidad cuando llegamos al lugar la muchacha y la señora nos hicieron sentir como en la casa nos dieron las normas del lugar y nos dejaron disfrutar sin problema la vista que tiene es amplia y dan ganas de quedarse viendo hacia el mar por horas la habitación cuenta con todo lo necesario el baño y servicio son espaciosos y cuentan con ducha no cuentan con parqueo propio pero tienen convenio con el 

In [59]:
#7.5 Escoger dos reseñas
texto1 = ejemplos.iloc[2]["reseña"]
texto2 = ejemplos.iloc[5]["reseña"]

In [60]:
#7.6 Comparar los embeddings
def comparar_contextos(texto1, texto2):

    emb1 = obtener_embedding(texto1)
    emb2 = obtener_embedding(texto2)

    similitud = cosine_similarity(
        emb1.reshape(1,-1),
        emb2.reshape(1,-1)
    )[0][0]

    print("="*70)

    print("RESEÑA 1\n")
    print(texto1)

    print("\n")

    print("RESEÑA 2\n")
    print(texto2)

    print("\n")

    print("Similitud coseno:", round(similitud,4))

    if similitud > 0.75:
        print("\nBETO considera ambos contextos muy similares.")

    elif similitud > 0.55:
        print("\nBETO detecta diferencias moderadas entre los contextos.")

    else:
        print("\nBETO considera que los contextos son bastante distintos.")

In [61]:
comparar_contextos(texto1, texto2)

RESEÑA 1

a mi me llamó mucho la atención desde el momento en que hice consultas y la muchacha me atendió con mucha amabilidad cuando llegamos al lugar la muchacha y la señora nos hicieron sentir como en la casa nos dieron las normas del lugar y nos dejaron disfrutar sin problema la vista que tiene es amplia y dan ganas de quedarse viendo hacia el mar por horas la habitación cuenta con todo lo necesario el baño y servicio son espaciosos y cuentan con ducha no cuentan con parqueo propio pero tienen convenio con el super que está al lado para mayor seguridad


RESEÑA 2

las habitaciones estás muy aseadas excelente presentación un poco pequeñas pero por el precio está muy bien justificado el servicio fue increíble la atención del host es muy detallada la ubicación me dió tranquilidad porque estamos a escasos 500mts de la playa y del centro super recomendado


Similitud coseno: 0.954

BETO considera ambos contextos muy similares.


La palabra "atención" fue evaluada en dos contextos diferentes. En la primera reseña se utiliza para describir la amabilidad y el trato recibido por parte del personal durante toda la estadía, mientras que en la segunda hace referencia a la atención brindada por el anfitrión del hospedaje. BETO obtuvo una similitud coseno de 0.9540, lo que indica que considera ambos usos muy similares desde el punto de vista semántico. Esto se debe a que, aunque las situaciones descritas son distintas, en ambos casos la palabra se relaciona con la calidad del servicio al cliente dentro del contexto turístico.

7. Búsqueda semántica

In [80]:
#8.1 Consulta
consulta = "hotel con piscina"

#Otras posibles consultas
#lugar tranquilo rodeado de naturaleza
#restaurante con buena comida

In [63]:
#8.2 Obtener embedding de la consulta
embedding_consulta = obtener_embedding(consulta)

In [64]:
#8.3 Calcular similitud con todas las reseñas
from sklearn.metrics.pairwise import cosine_similarity

similitudes = cosine_similarity(
    [embedding_consulta],
    list(df["embedding"])
)[0]

In [65]:
#8.4 Guardar la similitud
df["similitud"] = similitudes

In [67]:
#8.5 Mostrar las 5 mejores
top5 = df.sort_values(
    by="similitud",
    ascending=False
).head(5)

top5[[
    "nombre",
    "calificación",
    "reseña",
    "similitud"
]]

,nombre,calificación,reseña,similitud
371,rancho rio perlas resort spa,5,excelente el lugar la piscina de agua caliente...,0.862577
516,cabinas la sirenita,5,hotel de paso tienen habitaciones sencillas y ...,0.856554
1790,chicharronera casamoto,5,gran barbacoa,0.856403
38,guanacaste,5,lugar muy interesante con edificaciones estilo...,0.856055
5,alma de fuego,5,muy bonita experiencia lugar seguro familiar p...,0.855377


In [68]:
#8.6 Textualmente representado
for i, fila in top5.iterrows():

    print("="*90)
    print(f"Lugar: {fila['nombre']}")
    print(f"Calificación: {fila['calificación']}")
    print(f"Similitud: {fila['similitud']:.4f}")
    print()
    print(fila["reseña"])

Lugar: rancho rio perlas resort spa
Calificación: 5
Similitud: 0.8626

excelente el lugar la piscina de agua caliente perfecta
Lugar: cabinas la sirenita
Calificación: 5
Similitud: 0.8566

hotel de paso tienen habitaciones sencillas y con jacuzzi servicio de alimentos y bebidas a la habitación
Lugar: chicharronera casamoto
Calificación: 5
Similitud: 0.8564

gran barbacoa
Lugar: guanacaste
Calificación: 5
Similitud: 0.8561

lugar muy interesante con edificaciones estilo europeo acceso a la playa heladería restaurante cafetería y tienda
Lugar: alma de fuego
Calificación: 5
Similitud: 0.8554

muy bonita experiencia lugar seguro familiar piscina rancho con todo lo necesario patio cabaña con a c e internet todas las atracciones turísticas además de restaurantes super cerca visite con familia de 4 personas más que ideal


Se realizó una búsqueda semántica utilizando BETO. A partir de la consulta "hotel con piscina", el modelo generó un embedding y calculó la similitud coseno con todas las reseñas del corpus. Posteriormente se recuperaron las cinco reseñas con mayor similitud, demostrando la capacidad del modelo para identificar textos relacionados por significado y no únicamente por coincidencia exacta de palabras.

8. Masked Language Model (MLM)

In [69]:
#9.1 Cargar el pipeline
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model=modelo,
    tokenizer=tokenizer
)

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

In [70]:
#9.2 Escoger una frase
frase = "El servicio fue muy [MASK] durante toda la estadía."

In [71]:
#9.3 Obtener predicciones
predicciones = fill_mask(frase)

In [72]:
#9.4 Mostrar el Top 5
for i, p in enumerate(predicciones, 1):

    print(f"{i}. {p['token_str']} ({p['score']:.4f})")

1. bueno (0.3367)
2. efectivo (0.0509)
3. eficiente (0.0423)
4. satisfactorio (0.0382)
5. rápido (0.0325)


In [73]:
#Otro ejemplo
frase = "La habitación estaba muy [MASK]."

predicciones = fill_mask(frase)

for i, p in enumerate(predicciones,1):
    print(f"{i}. {p['token_str']} ({p['score']:.4f})")

1. limpia (0.2605)
2. fría (0.0808)
3. tranquila (0.0632)
4. sucia (0.0524)
5. bien (0.0411)


In [74]:
#Otro ejemplo
frase = "La comida estaba muy [MASK]."

predicciones = fill_mask(frase)

for i, p in enumerate(predicciones,1):
    print(f"{i}. {p['token_str']} ({p['score']:.4f})")

1. buena (0.7151)
2. rica (0.0945)
3. fría (0.0322)
4. bien (0.0319)
5. mala (0.0300)


Se utilizó la capacidad de Masked Language Modeling de BETO para predecir palabras ocultas dentro de diferentes frases relacionadas con el dominio turístico. En todos los casos el modelo propuso palabras coherentes con el contexto, como "excelente", "bueno" o "limpia", evidenciando que ha aprendido relaciones semánticas propias del idioma español.

Ejemplo Positivo vs Negativo

In [78]:
#Ejemplo Positivo
frase = "El servicio del hotel fue [MASK]."

predicciones = fill_mask(frase)

for i, p in enumerate(predicciones,1):
    print(f"{i}. {p['token_str']} ({p['score']:.4f})")

1. gratuito (0.3152)
2. cancelado (0.0757)
3. limitado (0.0315)
4. suspendido (0.0284)
5. interrumpido (0.0197)


In [79]:
#Ejemplo Negativo
frase = "El servicio del hotel fue [MASK] y nunca volvería."

predicciones = fill_mask(frase)

for i, p in enumerate(predicciones,1):
    print(f"{i}. {p['token_str']} ({p['score']:.4f})")

1. cancelado (0.2421)
2. suspendido (0.0548)
3. temporal (0.0445)
4. corto (0.0309)
5. terrible (0.0222)
